In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import kagglehub

path = kagglehub.dataset_download("hojjatk/mnist-dataset")

In [3]:
import os

print(os.listdir(path))
print("\n")
print(os.listdir(path + '/train-images-idx3-ubyte/'))

['t10k-images-idx3-ubyte', 't10k-images.idx3-ubyte', 't10k-labels-idx1-ubyte', 't10k-labels.idx1-ubyte', 'train-images-idx3-ubyte', 'train-images.idx3-ubyte', 'train-labels-idx1-ubyte', 'train-labels.idx1-ubyte']


['train-images-idx3-ubyte']


In [4]:
import numpy as np

def load_images(path):
    with open(path, 'rb') as f:
        magic, n, rows, cols = np.frombuffer(f.read(16), dtype='>i4')
        images = np.frombuffer(f.read(), dtype=np.uint8)
    return images.reshape(n, rows, cols)

def load_labels(path):
    with open(path, 'rb') as f:
        magic, n = np.frombuffer(f.read(8), dtype='>i4')
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

X_train = load_images(path + '/train-images-idx3-ubyte/train-images-idx3-ubyte')
y_train = load_labels(path + '/train-labels-idx1-ubyte/train-labels-idx1-ubyte')
X_test  = load_images(path + '/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
y_test  = load_labels(path + '/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

In [5]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler()
def process_x(X, fit):
    N, H, W = X.shape
    X_flat = X.reshape(N, H * W)
    if fit:
        scaler.fit(X_flat)
    return scaler.transform(X_flat)

X_train = process_x(X_train, True)
X_test = process_x(X_test, False)

In [6]:
# Show a few
from matplotlib import pyplot as plt
from ipywidgets import interact, IntSlider

interact(
    lambda i: plt.imshow(X_train[i].reshape(28, 28), cmap="gray"), i=IntSlider(min=0, max=len(y_train) - 1)
)

interactive(children=(IntSlider(value=0, description='i', max=59999), Output()), _dom_classes=('widget-interac…

<function __main__.<lambda>(i)>

In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, batch_size=256, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(X.astype(np.float32)),
        torch.from_numpy(y),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4)

train_loader = make_loader(X_train, y_train, shuffle=True)
test_loader = make_loader(X_test, y_test)

/tmp/ipykernel_63265/1608567004.py:7: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(y),


In [48]:
from SparseCAE import SparseCAE
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

model = SparseCAE(
    input_dim=28 * 28,
    latent_dim=4,
    hidden_dims=[256],
    lambda_sparse=1e-2,
    lambda_cae=1e-3,
    lr=1e-3,
)

trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    callbacks=[
        EarlyStopping(monitor="val/loss", patience=10),
        ModelCheckpoint(monitor="val/loss", filename="best", mode="min")
    ],
)
trainer.fit(model, train_loader, test_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode  | FLOPs
-------------------------------------------------------
0 | encoder | Sequential | 201 K  | train | 0    
1 | decoder | Sequential | 202 K  | train | 0    
-------------------------------------------------------
404 K     Trainable params
0         Non-trainable params
404 K     Total params
1.619     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [49]:
model.eval()

@torch.no_grad()
def reconstruct(i):
    x = torch.from_numpy(X_train[i:i+1].astype(np.float32))
    z     = model.encode(x).numpy().flatten()
    x_hat = model(x).numpy().reshape(28, 28)
    x_orig = X_train[i].reshape(28, 28)

    fig = plt.figure(figsize=(12, 4))

    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(x_orig, cmap="gray"); ax1.set_title("Original"); ax1.axis("off")

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(x_hat, cmap="gray"); ax2.set_title("Reconstruida"); ax2.axis("off")

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.bar(range(len(z)), z, color="steelblue")
    ax3.set_title("Espacio latente")
    ax3.set_xlabel("Dimensión")
    ax3.set_ylabel("Activación")
    ax3.set_ylim(0, 1)  # Sigmoid → [0, 1]
    ax3.axhline(0.5, color="red", linestyle="--", linewidth=0.8, alpha=0.5)

    plt.suptitle(f"Label: {y_train[i]}")
    plt.tight_layout()
    plt.show()

interact(reconstruct, i=IntSlider(min=0, max=len(y_train)-1, step=1))

interactive(children=(IntSlider(value=0, description='i', max=59999), Output()), _dom_classes=('widget-interac…

<function __main__.reconstruct(i)>

In [46]:
@torch.no_grad()
def reconstruct(i):
    x = torch.from_numpy(X_test[i:i+1].astype(np.float32))
    x_hat = model(x).numpy().reshape(28, 28)
    x_orig = X_test[i].reshape(28, 28)

    fig, axes = plt.subplots(1, 2, figsize=(5, 3))
    axes[0].imshow(x_orig, cmap="gray"); axes[0].set_title("Original");     axes[0].axis("off")
    axes[1].imshow(x_hat,  cmap="gray"); axes[1].set_title("Reconstruida"); axes[1].axis("off")
    plt.suptitle(f"Label: {y_test[i]}")
    plt.tight_layout()
    plt.show()

interact(reconstruct, i=IntSlider(min=0, max=len(y_test)-1, step=1))

interactive(children=(IntSlider(value=0, description='i', max=9999), Output()), _dom_classes=('widget-interact…

<function __main__.reconstruct(i)>

In [50]:
from ipywidgets import FloatSlider, HBox, VBox, Output
import ipywidgets as widgets

out = Output()
sliders = [FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description=f"z[{i}]") for i in range(4)]

def update(*args):
    z = torch.tensor([[s.value for s in sliders]], dtype=torch.float32)
    with torch.no_grad():
        x_hat = model.decoder(z).numpy().reshape(28, 28)
    with out:
        out.clear_output(wait=True)
        plt.figure(figsize=(3, 3))
        plt.imshow(x_hat, cmap="gray")
        plt.title(f"z = {[round(s.value, 2) for s in sliders]}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

for s in sliders:
    s.observe(update, names="value")

update()
display(VBox([*sliders, out]))